# 02. Preprocessing: Pipeline Design & Feature Scaling

## Preprocessing Strategy for EMBER Features
EMBER static PE features (2381 dimensions) contain a mixture of:
- **Structural counts and sizes** (e.g., number of sections, file size, export counts)
- **Byte histograms & byte-entropy histograms** (bounded 0 to 1 distributions)
- **Hashed feature strings** (FeatureHasher outputs for section names, DLL imports, exports)

### Crucial Preprocessing Principles:
1. **Prevent Data Leakage**: The preprocessor (scaler/imputer) must be **fit only on `X_train`** and then applied to `X_test`. Never fit on test data!
2. **Model-Specific Scaling**:
   - **Tree-based models** (Random Forest, LightGBM) are scale-invariant and do not strictly require feature scaling.
   - **Linear models** (Logistic Regression) require zero mean and unit variance (`StandardScaler`) for gradient/solver convergence.
3. **Pipeline Serialization**: Save the fitted preprocessor using `joblib` so inference on raw vectors replicates exact training transformations.


In [ ]:
import os
import joblib
import numpy as np
import ember
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = os.path.abspath(os.path.join(".."))
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "ember2018")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# 1. Load memory-mapped vectorized features
X_train, y_train, X_test, y_test = ember.read_vectorized_features(DATASET_DIR)
print(f"Loaded X_train shape: {X_train.shape}")
print(f"Loaded X_test shape:  {X_test.shape}")


## Building & Fitting the Preprocessing Pipeline
We build a `StandardScaler` fitted strictly on `X_train`.

In [ ]:
# Fit scaler exclusively on training data
preprocessor = StandardScaler()
print("[*] Fitting StandardScaler on X_train...")
X_train_scaled = preprocessor.fit_transform(X_train)

# Transform test data using the fitted training parameters
print("[*] Transforming X_test with fitted scaler...")
X_test_scaled = preprocessor.transform(X_test)

print("\nFeature Scaling Verification:")
print(f"  Original X_train mean: {np.mean(X_train):.4f}, std: {np.std(X_train):.4f}")
print(f"  Scaled X_train mean:   {np.mean(X_train_scaled):.4f}, std: {np.std(X_train_scaled):.4f}")
print(f"  Scaled X_test mean:    {np.mean(X_test_scaled):.4f}, std: {np.std(X_test_scaled):.4f}")


## Serializing the Preprocessing Artifact
Save the preprocessor to `models/preprocessor.pkl` for inference.

In [ ]:
preprocessor_path = os.path.join(MODELS_DIR, "preprocessor.pkl")
joblib.dump(preprocessor, preprocessor_path)
print(f"[+] Preprocessor successfully serialized to: {preprocessor_path}")
print(f"    File size: {os.path.getsize(preprocessor_path) / 1024:.2f} KB")

# Verify re-loading
loaded_scaler = joblib.load(preprocessor_path)
assert loaded_scaler.mean_.shape == (2381,), "Reload verification failed"
print("[+] Reload verification passed cleanly!")
